# Lab: Bag of Words and TF-IDF
**Course:** Natural Language Processing  |  **Topic:** Converting text to numbers

In earlier labs we cleaned text with NLTK (tokenization, stop words, stemming). Today we turn the cleaned text into numbers that a machine learning model can use.

**Flow of this lab:** raw text, then clean text, then Bag of Words matrix, then TF-IDF matrix, then compare documents.

Part 1 to Part 9 is a demo. Run each cell with Shift + Enter and read the explanation below it. The complete task for you to solve is at the end of the notebook.

## Part 1: Setup

In [25]:
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("Setup done")

Setup done


## Part 2: The dataset
Eight short SMS messages: 4 spam and 4 normal (ham). The data is small so that you can read every row.

In [26]:
messages = [
    "WINNER!! You have won a free prize. Call now to claim your reward",
    "Free entry in our weekly contest. Text WIN to 80085 now",
    "URGENT! Your account is locked. Verify now to avoid suspension",
    "Get a free ringtone. Reply YES to claim your free offer today",
    "Are we still meeting for lunch tomorrow?",
    "I will call you when I reach home",
    "Can you send me the notes from today's class?",
    "Don't forget to bring your laptop to the lab",
]
labels = ["spam", "spam", "spam", "spam", "ham", "ham", "ham", "ham"]

df = pd.DataFrame({"text": messages, "label": labels})
df

,text,label
0,WINNER!! You have won a free prize. Call now t...,spam
1,Free entry in our weekly contest. Text WIN to ...,spam
2,URGENT! Your account is locked. Verify now to ...,spam
3,Get a free ringtone. Reply YES to claim your f...,spam
4,Are we still meeting for lunch tomorrow?,ham
5,I will call you when I reach home,ham
6,Can you send me the notes from today's class?,ham
7,Don't forget to bring your laptop to the lab,ham


## Part 3: Cleaning the text (what we did before)
We apply the same steps as in the earlier labs: tokenize, lowercase, remove punctuation, remove stop words, stem. First on one message, step by step.

In [27]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

msg = df.text[0]
print("Original   :", msg)

tokens = word_tokenize(msg.lower())
print("Tokens     :", tokens)

words = [t for t in tokens if t.isalpha()]
print("Words only :", words)

no_stop = [t for t in words if t not in stop_words]
print("No stopword:", no_stop)

stems = [stemmer.stem(t) for t in no_stop]
print("Stemmed    :", stems)

Original   : WINNER!! You have won a free prize. Call now to claim your reward
Tokens     : ['winner', '!', '!', 'you', 'have', 'won', 'a', 'free', 'prize', '.', 'call', 'now', 'to', 'claim', 'your', 'reward']
Words only : ['winner', 'you', 'have', 'won', 'a', 'free', 'prize', 'call', 'now', 'to', 'claim', 'your', 'reward']
No stopword: ['winner', 'free', 'prize', 'call', 'claim', 'reward']
Stemmed    : ['winner', 'free', 'prize', 'call', 'claim', 'reward']


Now we put the same steps into one function, so we can use it on every message.

In [28]:
def clean_tokens(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return [stemmer.stem(t) for t in tokens]

def clean_text(text):
    return " ".join(clean_tokens(text))

df["clean"] = df.text.apply(clean_text)
df[["text", "clean"]]

,text,clean
0,WINNER!! You have won a free prize. Call now t...,winner free prize call claim reward
1,Free entry in our weekly contest. Text WIN to ...,free entri weekli contest text win
2,URGENT! Your account is locked. Verify now to ...,urgent account lock verifi avoid suspens
3,Get a free ringtone. Reply YES to claim your f...,get free rington repli ye claim free offer today
4,Are we still meeting for lunch tomorrow?,still meet lunch tomorrow
5,I will call you when I reach home,call reach home
6,Can you send me the notes from today's class?,send note today class
7,Don't forget to bring your laptop to the lab,forget bring laptop lab


## Part 4: Bag of Words with CountVectorizer
Idea: put the words of a document in a "bag". We ignore the order and only count how many times each word appears.

Steps: (1) build a vocabulary of all unique words, (2) count each word in each document. We first try it on three tiny sentences.

In [29]:
docs = ["the cat sat on the mat",
        "the dog sat on the log",
        "the cat chased the dog"]

cv = CountVectorizer()
X = cv.fit_transform(docs)

print("Vocabulary (word: column number):")
print(cv.vocabulary_)
print()
print("Column names:", list(cv.get_feature_names_out()))

Vocabulary (word: column number):
{'the': 7, 'cat': 0, 'sat': 6, 'on': 5, 'mat': 4, 'dog': 2, 'log': 3, 'chased': 1}

Column names: ['cat', 'chased', 'dog', 'log', 'mat', 'on', 'sat', 'the']


In [30]:
bow = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=["D1", "D2", "D3"])
bow

,cat,chased,dog,log,mat,on,sat,the
D1,1,0,0,0,1,1,1,2
D2,0,0,1,1,0,1,1,2
D3,1,1,1,0,0,0,0,2


How to read the table: each row is one document, each column is one word, and each number is how many times that word appears in the document. The word "the" appears twice in every document.

`fit_transform` does two things: it learns the vocabulary (fit) and then counts the words (transform).

## Part 5: Bag of Words on our messages
Now we use the cleaned messages from Part 3.

In [32]:
cv_msg = CountVectorizer()
X_msg = cv_msg.fit_transform(df.clean)

print("Matrix shape (messages x words):", X_msg.shape)

bow_msg = pd.DataFrame(X_msg.toarray(), columns=cv_msg.get_feature_names_out())
bow_msg

Matrix shape (messages x words): (8, 36)


,account,avoid,bring,call,claim,class,contest,entri,forget,free,get,home,lab,laptop,lock,lunch,meet,note,offer,prize,reach,repli,reward,rington,send,still,suspens,text,today,tomorrow,urgent,verifi,weekli,win,winner,ye
0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0
2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0
3,0,0,0,0,1,0,0,0,0,2,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0
7,0,0,1,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Most cells are zero. This is normal: each message uses only a few words out of the whole vocabulary. A matrix like this is called a sparse matrix.

## Part 6: Feature engineering (choosing the columns)
`CountVectorizer` has options that change the vocabulary. The table shows how the vocabulary size changes with each option on the raw messages.

In [33]:
options = {
    "default":                 CountVectorizer(),
    "stop_words='english'":    CountVectorizer(stop_words="english"),
    "ngram_range=(1, 2)":      CountVectorizer(ngram_range=(1, 2)),
    "min_df=2":                CountVectorizer(min_df=2),
}

rows = []
for name, vec in options.items():
    Xo = vec.fit_transform(df.text)
    rows.append((name, Xo.shape[1]))

pd.DataFrame(rows, columns=["option", "vocabulary size"])

,option,vocabulary size
0,default,56
1,stop_words='english',36
2,"ngram_range=(1, 2)",120
3,min_df=2,9


- `stop_words='english'` removes common words, so the vocabulary gets smaller.
- `ngram_range=(1, 2)` also adds word pairs such as "free prize", so the vocabulary gets bigger.
- `min_df=2` keeps only words that appear in at least 2 messages, so the vocabulary gets much smaller.

## Part 7: The problem with counts
Let us see which words appear most often in our cleaned messages.

In [34]:
totals = pd.Series(X_msg.toarray().sum(axis=0), index=cv_msg.get_feature_names_out())
totals.sort_values(ascending=False).head(8)

,0
free,4
call,2
claim,2
today,2
bring,1
avoid,1
account,1
contest,1


Words that are frequent are not always the most useful words. A word that appears in every message does not help us tell one message from another. We want a score that gives a high weight to words that are special to a document. This is TF-IDF.

## Part 8: TF-IDF
TF-IDF = TF x IDF

- **TF (term frequency):** how often the word appears in this document.
- **IDF (inverse document frequency):** how rare the word is across all documents. A word in every document gets a low IDF. A word in only one document gets a high IDF.

Textbook formulas: TF = count / number of words in the document, and IDF = log10(N / df), where N is the number of documents and df is the number of documents that contain the word.

Example with our 3 sentences (N = 3): the word "the" is in all 3 documents, so IDF = log10(3/3) = 0. The word "chased" is in only 1 document, so IDF = log10(3/1) = 0.477. So "the" gets weight 0 and "chased" gets a high weight.

Now the same idea with `TfidfVectorizer`.

In [35]:
tv = TfidfVectorizer()
T = tv.fit_transform(docs)

tfidf = pd.DataFrame(T.toarray(), columns=tv.get_feature_names_out(), index=["D1", "D2", "D3"])
tfidf.round(2)

,cat,chased,dog,log,mat,on,sat,the
D1,0.37,0.00,0.00,0.00,0.49,0.37,0.37,0.58
D2,0.00,0.00,0.37,0.49,0.00,0.37,0.37,0.58
D3,0.40,0.53,0.40,0.00,0.00,0.00,0.00,0.63


In [36]:
idf_values = pd.Series(tv.idf_, index=tv.get_feature_names_out())
idf_values.round(2).sort_values()

,0
the,1.00
cat,1.29
on,1.29
dog,1.29
sat,1.29
chased,1.69
log,1.69
mat,1.69


The words with the lowest IDF ("the") appear in many documents. The words with the highest IDF ("chased", "log", "mat") appear in only one document.

Note: scikit-learn uses a slightly different formula from the textbook (it smooths the IDF and normalizes each row), so the numbers are not exactly the same as the hand calculation, but the idea is the same.

## Part 9: BoW and TF-IDF on our messages, and document similarity
We compare the top 3 words of a message under BoW and under TF-IDF.

In [37]:
cv_s = CountVectorizer(analyzer=clean_tokens)
Xc = cv_s.fit_transform(df.text)

tv_s = TfidfVectorizer(analyzer=clean_tokens)
Xt = tv_s.fit_transform(df.text)

names = tv_s.get_feature_names_out()

def top_words(row, k=3):
    values = row.toarray().ravel()
    best = values.argsort()[::-1][:k]
    return [(names[i], round(float(values[i]), 2)) for i in best]

for i in [0, 3, 6]:
    print(df.text[i])
    print("   BoW top 3   :", top_words(Xc[i]))
    print("   TF-IDF top 3:", top_words(Xt[i]))
    print()

WINNER!! You have won a free prize. Call now to claim your reward
   BoW top 3   : [('winner', 1.0), ('claim', 1.0), ('reward', 1.0)]
   TF-IDF top 3: [('winner', 0.45), ('prize', 0.45), ('reward', 0.45)]

Get a free ringtone. Reply YES to claim your free offer today
   BoW top 3   : [('free', 2.0), ('ye', 1.0), ('today', 1.0)]
   TF-IDF top 3: [('free', 0.5), ('ye', 0.34), ('repli', 0.34)]

Can you send me the notes from today's class?
   BoW top 3   : [('class', 1.0), ('note', 1.0), ('send', 1.0)]
   TF-IDF top 3: [('note', 0.52), ('class', 0.52), ('send', 0.52)]



Cosine similarity tells us how similar two documents are. It is 1 for identical documents and 0 for documents that share no words.

In [38]:
sim = cosine_similarity(Xt)
pd.DataFrame(sim.round(2))

,0,1,2,3,4,5,6,7
0,1.00,0.10,0.0,0.27,0.0,0.19,0.00,0.0
1,0.10,1.00,0.0,0.15,0.0,0.00,0.00,0.0
2,0.00,0.00,1.0,0.00,0.0,0.00,0.00,0.0
3,0.27,0.15,0.0,1.00,0.0,0.00,0.13,0.0
4,0.00,0.00,0.0,0.00,1.0,0.00,0.00,0.0
5,0.19,0.00,0.0,0.00,0.0,1.00,0.00,0.0
6,0.00,0.00,0.0,0.13,0.0,0.00,1.00,0.0
7,0.00,0.00,0.0,0.00,0.0,0.00,0.00,1.0


In [39]:
sim_copy = sim.copy()
np.fill_diagonal(sim_copy, -1)          # ignore a message compared with itself
i, j = np.unravel_index(sim_copy.argmax(), sim_copy.shape)

print("Most similar pair:")
print(df.text[i])
print(df.text[j])
print("Similarity:", round(sim_copy[i, j], 2))

Most similar pair:
WINNER!! You have won a free prize. Call now to claim your reward
Get a free ringtone. Reply YES to claim your free offer today
Similarity: 0.27


## Summary
- Computers need numbers, so we convert each document into a vector.
- Bag of Words: build a vocabulary and count the words. Word order is ignored.
- `CountVectorizer` builds the BoW matrix. Its options (stop words, n-grams, min_df) change the columns.
- TF-IDF gives a high weight to words that are special to a document and a low weight to words found everywhere.
- Cosine similarity compares two document vectors.
- Cleaning the text first (earlier labs) gives a smaller and better vocabulary.

---
# Complete Task (to be solved by the student)

You are given 10 short product reviews: 5 positive and 5 negative. Using the code from the demo above, complete all the steps below. Write your code in the empty cells and your answers in the text cells.

Run the next cell first to load the data.

In [24]:
reviews = [
    "I love this phone, the battery lasts all day",
    "Great camera and very fast delivery",
    "The screen is bright and the sound is great",
    "Excellent quality, I love it",
    "Very happy with this phone, works great",
    "The battery is terrible and drains fast",
    "Poor quality, the screen broke in a week",
    "Delivery was slow and the box was damaged",
    "I hate this phone, the camera is bad",
    "Terrible sound and very slow performance",
]
review_labels = ["positive"] * 5 + ["negative"] * 5

task_df = pd.DataFrame({"text": reviews, "label": review_labels})
task_df

review_labels = ["positive"] * 5 + ["negative"] * 5

task_df = pd.DataFrame({"text": reviews, "label": review_labels})
task_df





stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()


reviews= df.text[0]
print("Original   :",
reviews)

tokens = word_tokenize(
reviews.lower())
print("Tokens     :", tokens)
words = [t for t in tokens if t.isalpha()]
print("Words only :", words)

no_stop = [t for t in words if t not in stop_words]
print("No stopword:", no_stop)

stems = [stemmer.stem(t) for t in no_stop]
print("Stemmed    :", stems)



Original   : WINNER!! You have won a free prize. Call now to claim your reward
Tokens     : ['winner', '!', '!', 'you', 'have', 'won', 'a', 'free', 'prize', '.', 'call', 'now', 'to', 'claim', 'your', 'reward']
Words only : ['winner', 'you', 'have', 'won', 'a', 'free', 'prize', 'call', 'now', 'to', 'claim', 'your', 'reward']
No stopword: ['winner', 'free', 'prize', 'call', 'claim', 'reward']
Stemmed    : ['winner', 'free', 'prize', 'call', 'claim', 'reward']


### Step 1: Clean the reviews
Apply the `clean_text` function from Part 3 to every review and store the result in a new column called `clean`. Display the `text` and `clean` columns.

In [40]:
task_df['clean'] = task_df.text.apply(clean_text)
display(task_df[['text', 'clean']])

,text,clean
0,"I love this phone, the battery lasts all day",love phone batteri last day
1,Great camera and very fast delivery,great camera fast deliveri
2,The screen is bright and the sound is great,screen bright sound great
3,"Excellent quality, I love it",excel qualiti love
4,"Very happy with this phone, works great",happi phone work great
5,The battery is terrible and drains fast,batteri terribl drain fast
6,"Poor quality, the screen broke in a week",poor qualiti screen broke week
7,Delivery was slow and the box was damaged,deliveri slow box damag
8,"I hate this phone, the camera is bad",hate phone camera bad
9,Terrible sound and very slow performance,terribl sound slow perform


### Step 2: Bag of Words
Use `CountVectorizer` on the cleaned reviews. Print the vocabulary, print the shape of the matrix, and display the matrix as a DataFrame with the words as column names.

In [41]:
cv_task = CountVectorizer()
X_task = cv_task.fit_transform(task_df.clean)

print("Vocabulary (word: column number):")
print(cv_task.vocabulary_)
print()
print("Matrix shape (reviews x words):", X_task.shape)
print()
bow_task = pd.DataFrame(X_task.toarray(), columns=cv_task.get_feature_names_out())
bow_task

Vocabulary (word: column number):
{'love': 16, 'phone': 18, 'batteri': 1, 'last': 15, 'day': 7, 'great': 12, 'camera': 5, 'fast': 11, 'deliveri': 8, 'screen': 21, 'bright': 3, 'sound': 23, 'excel': 10, 'qualiti': 20, 'happi': 13, 'work': 26, 'terribl': 24, 'drain': 9, 'poor': 19, 'broke': 4, 'week': 25, 'slow': 22, 'box': 2, 'damag': 6, 'hate': 14, 'bad': 0, 'perform': 17}

Matrix shape (reviews x words): (10, 27)



,bad,batteri,box,bright,broke,camera,damag,day,deliveri,drain,excel,fast,great,happi,hate,last,love,perform,phone,poor,qualiti,screen,slow,sound,terribl,week,work
0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,1
5,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
6,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,1,0
7,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
8,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0,0


### Step 3: Feature options
Build a table of vocabulary sizes for the raw review text using three settings: the default, `stop_words='english'`, and `ngram_range=(1, 2)`.

In [42]:
options_task = {
    "default":                 CountVectorizer(),
    "stop_words='english'":    CountVectorizer(stop_words="english"),
    "ngram_range=(1, 2)":      CountVectorizer(ngram_range=(1, 2)),
}

rows_task = []
for name, vec in options_task.items():
    Xo_task = vec.fit_transform(task_df.text)
    rows_task.append((name, Xo_task.shape[1]))

pd.DataFrame(rows_task, columns=["option", "vocabulary size"])

,option,vocabulary size
0,default,37
1,stop_words='english',27
2,"ngram_range=(1, 2)",89


### Step 4: TF-IDF
Use `TfidfVectorizer` on the cleaned reviews. Display the TF-IDF matrix as a DataFrame rounded to 2 decimals. Then print the 5 words with the lowest IDF and the 5 words with the highest IDF.

In [43]:
tv_task = TfidfVectorizer()
T_task = tv_task.fit_transform(task_df.clean)

tfidf_task = pd.DataFrame(T_task.toarray(), columns=tv_task.get_feature_names_out())
display(tfidf_task.round(2))

idf_values_task = pd.Series(tv_task.idf_, index=tv_task.get_feature_names_out())
print("\n5 words with the lowest IDF:")
print(idf_values_task.sort_values().head(5))
print("\n5 words with the highest IDF:")
print(idf_values_task.sort_values(ascending=False).head(5))

,bad,batteri,box,bright,broke,camera,damag,day,deliveri,drain,excel,fast,great,happi,hate,last,love,perform,phone,poor,qualiti,screen,slow,sound,terribl,week,work
0,0.00,0.43,0.00,0.00,0.00,0.00,0.00,0.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.5,0.43,0.00,0.37,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,0.00,0.00,0.00,0.00,0.00,0.52,0.00,0.0,0.52,0.00,0.00,0.52,0.45,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.00,0.58,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.43,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.49,0.00,0.49,0.00,0.00,0.00
3,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.64,0.00,0.00,0.00,0.00,0.0,0.54,0.00,0.00,0.00,0.54,0.00,0.00,0.00,0.00,0.00,0.00
4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.42,0.57,0.00,0.0,0.00,0.00,0.42,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.57
5,0.00,0.48,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.56,0.00,0.48,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.48,0.00,0.00
6,0.00,0.00,0.00,0.00,0.47,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.47,0.40,0.40,0.00,0.00,0.00,0.47,0.00
7,0.00,0.00,0.54,0.00,0.00,0.00,0.54,0.0,0.46,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.46,0.00,0.00,0.00,0.00
8,0.55,0.00,0.00,0.00,0.00,0.47,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.55,0.0,0.00,0.00,0.41,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.56,0.00,0.00,0.00,0.00,0.48,0.48,0.48,0.00,0.00



5 words with the lowest IDF:
great       2.011601
phone       2.011601
deliveri    2.299283
batteri     2.299283
fast        2.299283
dtype: float64

5 words with the highest IDF:
bad       2.704748
box       2.704748
bright    2.704748
excel     2.704748
broke     2.704748
dtype: float64


### Step 5: Compare BoW and TF-IDF
For review 0, review 5 and review 8, print the top 3 words under BoW and under TF-IDF (use the same idea as `top_words` in Part 9).

In [44]:
cv_task_s = CountVectorizer(analyzer=clean_tokens)
Xc_task = cv_task_s.fit_transform(task_df.text)

tv_task_s = TfidfVectorizer(analyzer=clean_tokens)
Xt_task = tv_task_s.fit_transform(task_df.text)

names_task = tv_task_s.get_feature_names_out()

def top_words_task(row, k=3):
    values = row.toarray().ravel()
    best = values.argsort()[::-1][:k]
    return [(names_task[i], round(float(values[i]), 2)) for i in best]

for i in [0, 5, 8]:
    print(task_df.text[i])
    print("   BoW top 3   :", top_words_task(Xc_task[i]))
    print("   TF-IDF top 3:", top_words_task(Xt_task[i]))
    print()

I love this phone, the battery lasts all day
   BoW top 3   : [('love', 1.0), ('phone', 1.0), ('last', 1.0)]
   TF-IDF top 3: [('last', 0.5), ('day', 0.5), ('love', 0.43)]

The battery is terrible and drains fast
   BoW top 3   : [('terribl', 1.0), ('fast', 1.0), ('drain', 1.0)]
   TF-IDF top 3: [('drain', 0.56), ('fast', 0.48), ('terribl', 0.48)]

I hate this phone, the camera is bad
   BoW top 3   : [('phone', 1.0), ('camera', 1.0), ('hate', 1.0)]
   TF-IDF top 3: [('hate', 0.55), ('bad', 0.55), ('camera', 0.47)]



### Step 6: Similarity
Calculate the cosine similarity between all reviews using the TF-IDF matrix. Find and print the most similar pair of reviews and their similarity score.

In [45]:
sim_task = cosine_similarity(Xt_task)
# Display the similarity matrix for all reviews
# pd.DataFrame(sim_task.round(2))

sim_task_copy = sim_task.copy()
np.fill_diagonal(sim_task_copy, -1)  # Ignore a review compared with itself
i, j = np.unravel_index(sim_task_copy.argmax(), sim_task_copy.shape)

print("Most similar pair of reviews:")
print(f"Review {i}: {task_df.text[i]}")
print(f"Review {j}: {task_df.text[j]}")
print("Similarity:", round(sim_task_copy[i, j], 2))

Most similar pair of reviews:
Review 1: Great camera and very fast delivery
Review 5: The battery is terrible and drains fast
Similarity: 0.25


### Step 7: Submission
Run the notebook from top to bottom (Runtime, Restart and run all) and submit it with all outputs and your answers.